In [80]:
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv
import os

load_dotenv()

True

In [81]:
# setting up the envirement variables
BIGQUERY_PROJECT = os.getenv('BIGQUERY_PROJECT')

In [82]:
# Connecting with bigquery to retrive the data from the tables
client = bigquery.Client(project=BIGQUERY_PROJECT)

In [83]:
# Extracting the results table from bigquery
fact_race_results_query = client.query("SELECT * FROM f1_data.fact_race_results")

In [84]:
# turning the query results into a dataframe
fact_race_results = fact_race_results_query.to_dataframe() 

In [85]:
fact_race_results.head()

,result_id,DriverId,TeamId,circuit_id,year,round_number,Position,GridPosition,Points,Laps,race_time_seconds,gap_to_winner_seconds,Status,ClassifiedPosition,is_classified,session_type
0,2020_16_89,aitken,williams,sakhir_grand_prix,2020,16,16,17,0,87,33.674,-5441.440,Finished,16,True,R
1,2019_12_23,albon,toro_rosso,hungarian_grand_prix,2019,12,10,12,1,69,NaN,NaN,+1 Lap,10,True,R
2,2020_10_23,albon,red_bull,russian_grand_prix,2020,10,10,15,1,53,97.860,-5542.504,Finished,10,True,R
3,2022_3_23,albon,williams,australian_grand_prix,2022,3,10,20,1,58,79.382,-5187.166,Finished,10,True,R
4,2022_14_23,albon,williams,belgian_grand_prix,2022,14,10,6,1,44,101.900,-5050.994,Finished,10,True,R


In [86]:
fact_race_results.isnull().sum()

result_id                   0
DriverId                    0
TeamId                      0
circuit_id                  0
year                        0
round_number                0
Position                    0
GridPosition                3
Points                      0
Laps                        0
race_time_seconds        1153
gap_to_winner_seconds    1153
Status                      0
ClassifiedPosition          0
is_classified               0
session_type                0
dtype: int64

In [87]:
# Extracting laps table from bigquery
fact_lap_times_query = client.query("SELECT * FROM f1_data.fact_lap_times")

In [88]:
fact_lap_times = fact_lap_times_query.to_dataframe()

In [89]:
fact_lap_times.head()

,lap_id,DriverId,circuit_id,year,round_number,lap_number,stint,lap_time_seconds,sector1_seconds,sector2_seconds,...,speedSt,compound,tyre_life,fresh_tyre,is_pit_lap,is_valid_lap,is_personal_best,Position,track_status,session_type
0,2018_21_ALO_24,alonso,abu_dhabi_grand_prix,2018,21,24,1,106.905,18.356,45.269,...,299.0,ULTRASOFT,24,True,False,True,False,10,1,R
1,2018_21_HAR_49,brendon_hartley,abu_dhabi_grand_prix,2018,21,49,2,105.099,18.187,44.757,...,300.0,SUPERSOFT,47,True,False,True,False,12,1,R
2,2018_21_SIR_48,sirotkin,abu_dhabi_grand_prix,2018,21,48,2,105.223,18.118,44.811,...,307.0,ULTRASOFT,13,True,False,True,False,15,21,R
3,2018_21_PER_45,perez,abu_dhabi_grand_prix,2018,21,45,2,103.983,18.071,43.955,...,307.0,SUPERSOFT,19,True,False,True,False,8,1,R
4,2018_21_VER_17,max_verstappen,abu_dhabi_grand_prix,2018,21,17,1,106.678,18.353,44.441,...,295.0,HYPERSOFT,20,False,True,False,False,2,1,R


In [90]:
# the null values in the laps table are due to the fact that some drivers did not finish
fact_lap_times.isnull().sum()

lap_id                 0
DriverId               0
circuit_id             0
year                   0
round_number           0
lap_number             0
stint                646
lap_time_seconds    3156
sector1_seconds     3997
sector2_seconds      376
sector3_seconds      535
speedI1              283
speedI2             1170
speedFl             6435
speedSt              361
compound               0
tyre_life           1361
fresh_tyre             0
is_pit_lap             0
is_valid_lap           0
is_personal_best     210
Position             281
track_status           0
session_type           0
dtype: int64

# The goal that i'm trying to reach is building a model that can predict if a driver has been DNF or has finished the race, looking at the data, most of null rows in lap times belongs to drivers that has been DNF, most of features that i need are in the lap times table, so i have to join the result that have the target with lap times table that have the features

In [91]:
fact_race_results.columns

Index(['result_id', 'DriverId', 'TeamId', 'circuit_id', 'year', 'round_number',
       'Position', 'GridPosition', 'Points', 'Laps', 'race_time_seconds',
       'gap_to_winner_seconds', 'Status', 'ClassifiedPosition',
       'is_classified', 'session_type'],
      dtype='object')

In [92]:
fact_lap_times.columns

Index(['lap_id', 'DriverId', 'circuit_id', 'year', 'round_number',
       'lap_number', 'stint', 'lap_time_seconds', 'sector1_seconds',
       'sector2_seconds', 'sector3_seconds', 'speedI1', 'speedI2', 'speedFl',
       'speedSt', 'compound', 'tyre_life', 'fresh_tyre', 'is_pit_lap',
       'is_valid_lap', 'is_personal_best', 'Position', 'track_status',
       'session_type'],
      dtype='object')

# as we can see we can join the two tables using DriverId, then we have to aggregate lap times on race table level, bcs if we do the opposite will have each lap with a status of dnf or finished

In [93]:
# here we are joining the two tables to get the lap times for each driver in each race and extracting some features from the lap times table to use in our model

query = """SELECT race.DriverId, race.year, race.round_number, race.GridPosition, race.circuit_id, race.TeamId ,AVG(lap_time_seconds) AS lap_time_seconds_total, STDDEV(lap_time_seconds) as std_lap_times, MAX(tyre_life) as tyre_life,
SUM(CASE WHEN is_pit_lap then 1 ELSE 0 END) pit_laps, MAX(track_status) As track_status, MAX(lap_number) as last_lap_number, Status  FROM `f1_data.fact_race_results` race
JOIN `f1-data-pipeline-497719.f1_data.fact_lap_times` laps USING(DriverId, year, round_number)
GROUP BY race.DriverId, race.year, race.round_number, race.GridPosition, race.circuit_id, race.TeamId, Status
ORDER BY race.DriverId"""

data = client.query(query)

df = data.to_dataframe()

In [94]:
df.shape

(3421, 13)

In [95]:
df.head()

,DriverId,year,round_number,GridPosition,circuit_id,TeamId,lap_time_seconds_total,std_lap_times,tyre_life,pit_laps,track_status,last_lap_number,Status
0,aitken,2020,16,17,sakhir_grand_prix,williams,63.319402,10.585356,31,6,71,87,Finished
1,albon,2024,10,20,spanish_grand_prix,williams,82.436554,3.738817,25,5,1,65,Lapped
2,albon,2020,11,5,eifel_grand_prix,red_bull,95.936304,5.541816,16,3,21,23,Radiator
3,albon,2022,3,20,australian_grand_prix,williams,92.171207,16.912831,57,2,671,58,Finished
4,albon,2022,14,6,belgian_grand_prix,williams,116.506357,4.941852,18,4,41,44,Finished


In [96]:
df.dtypes

DriverId                   object
year                        Int64
round_number                Int64
GridPosition                Int64
circuit_id                 object
TeamId                     object
lap_time_seconds_total    float64
std_lap_times             float64
tyre_life                   Int64
pit_laps                    Int64
track_status               object
last_lap_number             Int64
Status                     object
dtype: object

In [97]:
# we need to convert the track_status column to int64 type as it is currently of object type and we need to use it in our model
df['track_status'] = pd.to_numeric(df['track_status'], errors='coerce').astype('Int64')
df.dtypes

DriverId                   object
year                        Int64
round_number                Int64
GridPosition                Int64
circuit_id                 object
TeamId                     object
lap_time_seconds_total    float64
std_lap_times             float64
tyre_life                   Int64
pit_laps                    Int64
track_status                Int64
last_lap_number             Int64
Status                     object
dtype: object

In [98]:
df.isnull().sum()

DriverId                    0
year                        0
round_number                0
GridPosition                0
circuit_id                  0
TeamId                      0
lap_time_seconds_total     71
std_lap_times             112
tyre_life                   2
pit_laps                    0
track_status                1
last_lap_number             0
Status                      0
dtype: int64

In [99]:
df['Status'].unique()

array(['Finished', 'Lapped', 'Radiator', '+1 Lap', 'Electronics',
       'Collision damage', '+2 Laps', 'Retired', 'Hydraulics',
       'Mechanical', 'Collision', 'Withdrew', 'Wheel', 'Engine',
       'Exhaust', 'Rear wing', 'Accident', 'Gearbox', 'Water pump',
       'Water leak', 'Brakes', 'Water pressure', 'Power Unit',
       'Wheel nut', 'Spun off', 'Puncture', 'Damage', 'Cooling system',
       'Power loss', 'Steering', 'Suspension', 'Disqualified',
       'Transmission', '+3 Laps', 'Fuel pressure', 'Overheating',
       'Front wing', 'Illness', 'Oil leak', '+5 Laps', 'Undertray',
       'Turbo', 'Tyre', 'Fuel leak', 'Fuel pump', 'Out of fuel',
       'Battery', 'Electrical', 'Debris', 'Vibrations', 'Differential',
       '+6 Laps'], dtype=object)

# As we Discoverd above, we have some nulls values in lap_time_seconds_total and std_lap_times, we need to clean it before moving it into the model.

# We are going to drop lap_time_seconds_total rows from the dataset, then we are going to set null values in std_lap_times with 0

In [100]:
df = df.dropna(subset=['lap_time_seconds_total'])

In [101]:
df['std_lap_times'] = df['std_lap_times'].fillna(0)

In [102]:
df['tyre_life'] = df['tyre_life'].fillna(df['tyre_life'].median())

In [103]:
df.isnull().sum()

DriverId                  0
year                      0
round_number              0
GridPosition              0
circuit_id                0
TeamId                    0
lap_time_seconds_total    0
std_lap_times             0
tyre_life                 0
pit_laps                  0
track_status              0
last_lap_number           0
Status                    0
dtype: int64

In [104]:
df.head()

,DriverId,year,round_number,GridPosition,circuit_id,TeamId,lap_time_seconds_total,std_lap_times,tyre_life,pit_laps,track_status,last_lap_number,Status
0,aitken,2020,16,17,sakhir_grand_prix,williams,63.319402,10.585356,31,6,71,87,Finished
1,albon,2024,10,20,spanish_grand_prix,williams,82.436554,3.738817,25,5,1,65,Lapped
2,albon,2020,11,5,eifel_grand_prix,red_bull,95.936304,5.541816,16,3,21,23,Radiator
3,albon,2022,3,20,australian_grand_prix,williams,92.171207,16.912831,57,2,671,58,Finished
4,albon,2022,14,6,belgian_grand_prix,williams,116.506357,4.941852,18,4,41,44,Finished


In [105]:
df.shape

(3350, 13)

# Status column have several values showing that the driver has either finished a race or didn't, but the issue is it break down the the real issue why the driver has been dnf (e.g: engine, crash...), or if it finished the race (e.g: Finished, lapped...), we only want to know if the driver dnf status simple as finished or dnf, no need for the reason, so we use feature enginnering to turn these unnecessary values into just two: Finished = 0, DNF = 1.

In [106]:
finished_values = ['Finished', 'Lapped', '+1 Lap', '+2 Laps', '+3 Laps', '+5 Laps', '+6 Laps']

def label_dnf(status):
    if status in finished_values:
        return 0
    else:
        return 1

df['DNF'] = df['Status'].apply(label_dnf)

In [107]:
df.head()

,DriverId,year,round_number,GridPosition,circuit_id,TeamId,lap_time_seconds_total,std_lap_times,tyre_life,pit_laps,track_status,last_lap_number,Status,DNF
0,aitken,2020,16,17,sakhir_grand_prix,williams,63.319402,10.585356,31,6,71,87,Finished,0
1,albon,2024,10,20,spanish_grand_prix,williams,82.436554,3.738817,25,5,1,65,Lapped,0
2,albon,2020,11,5,eifel_grand_prix,red_bull,95.936304,5.541816,16,3,21,23,Radiator,1
3,albon,2022,3,20,australian_grand_prix,williams,92.171207,16.912831,57,2,671,58,Finished,0
4,albon,2022,14,6,belgian_grand_prix,williams,116.506357,4.941852,18,4,41,44,Finished,0


In [108]:
# Now that we have created the DNF column, we can drop the Status column as it is no longer needed.
df = df.drop(columns=['Status'])

In [109]:
df.head()

,DriverId,year,round_number,GridPosition,circuit_id,TeamId,lap_time_seconds_total,std_lap_times,tyre_life,pit_laps,track_status,last_lap_number,DNF
0,aitken,2020,16,17,sakhir_grand_prix,williams,63.319402,10.585356,31,6,71,87,0
1,albon,2024,10,20,spanish_grand_prix,williams,82.436554,3.738817,25,5,1,65,0
2,albon,2020,11,5,eifel_grand_prix,red_bull,95.936304,5.541816,16,3,21,23,1
3,albon,2022,3,20,australian_grand_prix,williams,92.171207,16.912831,57,2,671,58,0
4,albon,2022,14,6,belgian_grand_prix,williams,116.506357,4.941852,18,4,41,44,0


# Now the data is clean and ready to be feed to our model

In [110]:
df.isnull().sum()

DriverId                  0
year                      0
round_number              0
GridPosition              0
circuit_id                0
TeamId                    0
lap_time_seconds_total    0
std_lap_times             0
tyre_life                 0
pit_laps                  0
track_status              0
last_lap_number           0
DNF                       0
dtype: int64

In [111]:
# Checking the distribution of the target variable DNF, data is imbalanced with 80% of the drivers finishing, we need to use techniques like oversampling or undersampling to balance the data before training the model.
df['DNF'].value_counts(normalize=True)

DNF
0    0.873731
1    0.126269
Name: proportion, dtype: float64

In [112]:
df.shape

(3350, 13)

# Now the final step, saving data as parquet, ready for phase 2

In [113]:
df.to_parquet('../data/f1_dataset.parquet', index=False)